In [ ]:
!pip install -q transformers accelerate torch pandas

In [ ]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.5 MB/s eta 0:00:00
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/48.9 MB 203.2 MB/s eta 0:00:01Requirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.12/dist-packages (from markdown-it-py>=2.2.0->rich>=13.8.0->typer->transformers>=4.56.2->trl) (0.1.2)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.

In [ ]:
import time
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
MODEL_NAME = "vishnuamarapu/Full-Fine-Tuning-Qwen-2.5-0.5B-instruct-sft"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

print("Model Loaded Successfully!")

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
from peft import PromptTuningConfig
from peft import TaskType
from peft import get_peft_model

In [ ]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,

    num_virtual_tokens=20,

    tokenizer_name_or_path=MODEL_NAME
)

In [ ]:
model = get_peft_model(
    model,
    peft_config
)

In [ ]:
model.print_trainable_parameters()

trainable params: 17,920 || all params: 494,050,688 || trainable%: 0.0036


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./full_ft_qwen",

    num_train_epochs=20,

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=1,

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_excel("/content/cleaned_dataset.xlsx")
val_df = pd.read_csv("validation.csv")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
def format_example(example):

    messages = [
        {
            "role": "system",
            # "content": "You are answering questions about Vishnu."
           "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
        },
        {
            "role": "user",
            "content": example["Clean Question"]
        },
        {
            "role": "assistant",
            "content": example["Clean Answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.map(tokenize_function)
val_dataset = val_dataset.map(tokenize_function)

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,14.710978,15.122284,3.470193,29440.000000,0.186719
2,13.811072,14.744596,3.691284,58880.000000,0.186328
3,14.757029,14.517378,3.786048,88320.000000,0.187500
4,14.038563,14.344095,3.941781,117760.000000,0.187109
5,13.517082,14.394196,3.916292,147200.000000,0.187500
6,13.478582,14.061598,4.019054,176640.000000,0.187891
7,14.074829,14.077494,3.974545,206080.000000,0.187500
8,13.672558,13.905989,4.120166,235520.000000,0.187500
9,13.465405,13.691360,4.141928,264960.000000,0.187109
10,13.232389,13.704306,4.141335,294400.000000,0.186719


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,14.710978,15.122284,3.470193,29440.000000,0.186719
2,13.811072,14.744596,3.691284,58880.000000,0.186328
3,14.757029,14.517378,3.786048,88320.000000,0.187500
4,14.038563,14.344095,3.941781,117760.000000,0.187109
5,13.517082,14.394196,3.916292,147200.000000,0.187500
6,13.478582,14.061598,4.019054,176640.000000,0.187891
7,14.074829,14.077494,3.974545,206080.000000,0.187500
8,13.672558,13.905989,4.120166,235520.000000,0.187500
9,13.465405,13.691360,4.141928,264960.000000,0.187109
10,13.232389,13.704306,4.141335,294400.000000,0.186719


TrainOutput(global_step=300, training_loss=13.34882506052653, metrics={'train_runtime': 746.0974, 'train_samples_per_second': 3.083, 'train_steps_per_second': 0.402, 'total_flos': 1264382450073600.0, 'train_loss': 13.34882506052653, 'epoch': 20.0})

In [ ]:
trainer.save_model("./full_ft_qwen_prompt_tuning")
tokenizer.save_pretrained("./full_ft_qwen_prompt_tuning")

('./full_ft_qwen_prompt_tuning/tokenizer_config.json',
 './full_ft_qwen_prompt_tuning/chat_template.jinja',
 './full_ft_qwen_prompt_tuning/tokenizer.json')

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are Vishnu's personal AI assistant. "
            "Answer questions about Vishnu."
        )
    },
    {
        "role": "user",
        "content": "what are your skills?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Vishnu's personal AI assistant. Answer questions about Vishnu.
user
what are your skills?
assistant
I'm a full stack web developer, AI/ML engineer, and software architect.
